# Surge Control Device Verification

Mirrors **`tests/test_surge_device_verification.py`** and related surge regressions. Passive devices are shown in **both** transient roles they are used for in production:

| Device | Overpressure (valve closure) | Low pressure (pump trip) | Other |
|--------|------------------------------|--------------------------|-------|
| **Standpipe** | §2 B.8 + §4 valve-side (`test_surge_device_mitigation`) | §5 pump trip (mitigation) | §3 TSNet §B.8.5 |
| **HydropneumaticTank** | §4 fast closure | §5 pump trip + §8 sizing link | — |
| **AirValve** | — | §6 trip vacuum floor | §7 restart / trapped air |

> **Sizing sweeps:** [`surge_design_rules_verification.ipynb`](surge_design_rules_verification.ipynb)

[![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/jlillywh/RTHYM-MOC/main?labpath=examples%2Fsurge_device_verification.ipynb)

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rthym_moc
from _verification_notebook_setup import bootstrap_verification_notebook

REPO_ROOT, TESTS_DIR = bootstrap_verification_notebook()
print(f"Repository root: {REPO_ROOT}")
print(f"rthym_moc: {getattr(rthym_moc, '__version__', 'unknown')}")
from surge_device_verification_utils import (
    SP_H_JOUK_PEAK_FT, SP_H_PEAK_ANALYTICAL_FT, SP_H_SS_FT, SP_T_OSC_S,
    TRIP_START_S, TRIP_END_S, TSNET_PEAK_DIFF_FT, TSNET_RMS_0_20_FT,
    evaluate_air_valve_restart, evaluate_air_valve_vs_unprotected,
    evaluate_hydropneumatic_precharge, evaluate_standpipe,
    evaluate_valve_closure_mitigation, try_tsnet_standpipe_overlay,
)
from surge_design_rules_verification_utils import evaluate_standpipe_sweep

## 2. Standpipe — B.8 valve closure (Joukowsky + mass oscillation)

In [ ]:
res_none, res_sp, sp_metrics = evaluate_standpipe()
t_none = np.asarray(res_none["time"])
t_sp = np.asarray(res_sp["time"])
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(t_none, res_none["node_head"]["J1"], "--", color="gray", label="No standpipe (J1)")
ax.plot(t_sp, res_sp["node_head"]["SP1"], color="teal", label="With standpipe (SP1)")
ax.axhline(SP_H_JOUK_PEAK_FT, color="seagreen", linestyle="-.", label="Joukowsky ref.")
ax.axhline(SP_H_PEAK_ANALYTICAL_FT, color="darkorange", linestyle=":", label="Mass-osc. ref.")
ax.set_xlim(0, SP_T_OSC_S / 2)
ax.set_ylabel("Head (ft)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.show()
print(f"Standpipe B.8 PASS={sp_metrics.passed}  mitigation={sp_metrics.mitigation_fraction:.1%}")

## 3. TSNet standpipe overlay (Appendix B.8.5)

Documented cross-engine agreement (not default CI). Set `RUN_TSNET = True` to re-run TSNet if installed.

In [ ]:
RUN_TSNET = False
overlay = try_tsnet_standpipe_overlay(run_tsnet=RUN_TSNET)
fig, ax = plt.subplots(figsize=(12, 4))
mask = overlay.rthym_time_s <= 20.0
ax.plot(overlay.rthym_time_s[mask], overlay.rthym_head_ft[mask], label="RTHYM-MOC SP1")
if RUN_TSNET and overlay.ran_tsnet and overlay.tsnet_time_s is not None:
    m = overlay.tsnet_time_s <= 20.0
    ax.plot(overlay.tsnet_time_s[m], overlay.tsnet_head_ft[m], "--", label="TSNet J1")
ax.set_title("Standpipe peak comparison (0–20 s)")
ax.set_ylabel("Head (ft)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()
print(f"Documented: peak diff {overlay.documented_peak_diff_ft:.2f} ft, RMS {overlay.documented_rms_ft:.2f} ft")
if overlay.ran_tsnet:
    print(f"This run: peak diff {overlay.peak_diff_ft:.2f} ft, RMS {overlay.rms_ft:.2f} ft")
elif overlay.tsnet_error:
    print(f"TSNet skipped/error: {overlay.tsnet_error}")

## 4. Valve-side protection on fast closure (`test_surge_device_mitigation`)

In [ ]:
base_valve, valve_cases, valve_metrics = evaluate_valve_closure_mitigation()
t_b = np.asarray(base_valve["time"])
h_b = np.asarray(base_valve["node_head"]["Prot"])
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(t_b, h_b, "k--", label="Unprotected")
colors = {"Standpipe": "teal", "HydropneumaticTank": "purple"}
for kind, color in colors.items():
    d = valve_cases[kind]
    ax.plot(d["time"], d["node_head"]["Prot"], color=color, label=kind)
ax.set_ylabel("Protected node head (ft)")
ax.set_title("Fast valve closure: standpipe and HPT at the valve")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()
for m in valve_metrics:
    print(f"  {m.kind}: peak={m.peak_head_ft:.0f} ft, cut={m.peak_reduction_ft:.0f} ft, PASS={m.passed}")

## 5. Hydropneumatic tank — pump trip

In [ ]:
res_none_hpt, res_hpt, hpt_trip, hpt_pre = evaluate_hydropneumatic_precharge()
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(res_none_hpt["time"], res_none_hpt["node_head"]["Jd"], "--", color="gray", label="Unprotected")
ax.plot(res_hpt["time"], res_hpt["node_head"]["Jd"], color="purple", label="HPT 10 ft³")
ax.axhline(hpt_trip.reference_floor_ft, color="orange", linestyle=":", label="Anchored floor")
ax.axvspan(TRIP_START_S, TRIP_END_S, color="gold", alpha=0.15)
ax.set_xlim(4.5, 8.5)
ax.set_ylabel("Jd head (ft)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()
print(f"HPT trip PASS={hpt_trip.passed}  improvement={hpt_trip.improvement_vs_none_ft:.1f} ft")

## 6. Air valve — pump trip vacuum floor

In [ ]:
res_base_av, res_av, av_metrics = evaluate_air_valve_vs_unprotected()
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(res_base_av["time"], res_base_av["node_head"]["Vent"], "--", color="gray", label="Unprotected")
ax.plot(res_av["time"], res_av["node_head"]["Vent"], color="steelblue", label="Air valve")
ax.axhline(-5.0, color="orange", linestyle=":", label="Protected floor")
ax.axvspan(TRIP_START_S, TRIP_END_S, color="gold", alpha=0.15)
ax.set_xlim(4.5, 8.5)
ax.set_ylabel("Vent head (ft)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()
print(f"Air-valve trip PASS={av_metrics.passed}")

## 7. Air valve — restart and trapped-air release (`test_air_valve`)

In [ ]:
res_restart, restart_metrics = evaluate_air_valve_restart()
t = np.asarray(res_restart["time"])
h = np.asarray(res_restart["node_head"]["Vent"])
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t, h, color="steelblue")
ax.axvline(5.0, color="gray", linestyle="--", alpha=0.5, label="Trip")
ax.axvline(8.0, color="green", linestyle="--", alpha=0.5, label="Restart")
ax.set_ylabel("Vent head (ft)")
ax.set_title("Trapped-air cushion then gradual release")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()
print(f"Restart PASS={restart_metrics.passed}")
print(f"  pre={restart_metrics.pretrip_head_ft:.1f} early={restart_metrics.early_restart_head_ft:.1f} late={restart_metrics.late_restart_head_ft:.1f} ft")

## 8. Why bigger tank / closer placement helps (preview)

In [ ]:
sweep = evaluate_standpipe_sweep()
areas = [p.area_ft2 for p in sweep]
peaks = [p.peak_head_ft for p in sweep]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(areas, peaks, "o-")
ax.set_xlabel("Standpipe area (ft²)")
ax.set_ylabel("Closure peak at SP1 (ft)")
ax.set_title("Larger standpipe → lower peak (see surge_design_rules notebook)")
ax.grid(True, alpha=0.3)
plt.show()

## 9. Summary

In [ ]:
rows = [
    ("Standpipe B.8", sp_metrics.passed),
    ("Valve-side SP/HPT", all(m.passed for m in valve_metrics)),
    ("HPT pump trip", hpt_trip.passed),
    ("Air valve trip", av_metrics.passed),
    ("Air valve restart", restart_metrics.passed),
]
for name, ok in rows:
    print(f"{name:<24} {'PASS' if ok else 'FAIL'}")
print("Overall:", "PASS" if all(r[1] for r in rows) else "FAIL")